In [30]:
!pip install -r requirements.txt

35252.21s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


# Semantic 3D Scene Graph Generation from Egocentric RGB-D Sequences with IFC Priors

This notebook implements a pipeline for generating semantic 3D scene graphs from egocentric RGB-D sequences, enriched with structural and topological priors from IFC (Industry Foundation Classes) data. The approach integrates visual perception with BIM (Building Information Modeling) knowledge to create actionable spatial representations for industrial robotics and spatial computing applications.

## Overview

The pipeline consists of the following stages:
1. IFC data parsing and prior extraction
2. RGB-D sequence processing and 3D reconstruction
3. Visual feature extraction and object detection
4. Initial scene graph construction from visual data
5. IFC prior integration and constraint application
6. Graph refinement and optimization
7. Visualization and validation

## Theoretical Justification

Scene graphs provide a structured representation of environments, capturing both geometric and semantic information. In industrial settings, IFC models offer rich structural priors that can constrain and enrich visual-only scene graphs. This multimodal fusion addresses limitations of purely visual approaches, such as occlusions, lighting variations, and semantic ambiguities.

Key challenges:
- Aligning IFC coordinate systems with RGB-D camera frames
- Handling dynamic objects not present in static BIM models
- Resolving conflicts between visual observations and IFC priors
- Computational efficiency for real-time applications

## 1. Import Required Libraries and Dependencies

In [32]:
!pip uninstall open3d -y

35270.96s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
Found existing installation: open3d 0.19.0
Uninstalling open3d-0.19.0:
  Successfully uninstalled open3d-0.19.0


In [33]:
!pip install --no-cache-dir open3d

35282.06s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 15.5 MB/s eta 0:00:0000:0100:01


In [1]:
import os
import json
import numpy as np
try:
    import open3d as o3d
except ImportError as e:
    print(f"Failed to import open3d: {e}")
    print("Open3D may not be compatible with Python 3.12. Try installing a compatible version, e.g., open3d==0.18.0")
    raise
import ifcopenshell
import networkx as nx
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R
import logging



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [5]:
# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configuration parameters
DATA_DIR = "Data"
DATA_DR2 = "Data/BasicHouse_with_pc"

IFC_FILE = os.path.join(DATA_DIR, "BasicHouse.ifc")
RGBD_DIR = os.path.join(DATA_DIR, "BasicHouse_with_pc")
CAMERA_INFO = os.path.join(DATA_DR2, "camera_info.json")

In [6]:
print(CAMERA_INFO)

Data/BasicHouse_with_pc/camera_info.json


In [7]:
import json

def load_intrinsics(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Extract values from the matrix
    intrinsics_matrix = data['intrinsics']
    
    return {
        'fx': intrinsics_matrix[0][0],
        'fy': intrinsics_matrix[1][1],
        'cx': intrinsics_matrix[0][2],
        'cy': intrinsics_matrix[1][2],
        'width': data['width'],
        'height': data['height']
    }

# Usage
CAMERA_INTRINSICS = load_intrinsics(CAMERA_INFO)
print(CAMERA_INTRINSICS)

{'fx': 667.2513908946974, 'fy': 667.2513908946974, 'cx': 512.0, 'cy': 384.0, 'width': 1024, 'height': 768}


## 2. Load and Parse IFC Data

We use ifcopenshell to parse the IFC file and extract building entities. IFC files contain hierarchical representations of building components with spatial relationships.

In [8]:
def load_ifc_model(ifc_path):
    """Load IFC model using ifcopenshell."""
    if not os.path.exists(ifc_path):
        raise FileNotFoundError(f"IFC file not found: {ifc_path}")
    model = ifcopenshell.open(ifc_path)
    logger.info("Loaded IFC model")
    return model

def extract_building_entities(model):
    """Extract building entities from IFC model."""
    entities = {}
    
    # Extract walls
    walls = model.by_type("IfcWall")
    entities['walls'] = [{'id': w.id(), 'name': w.Name, 'geometry': get_geometry(w)} for w in walls]
    
    # Extract floors
    floors = model.by_type("IfcSlab")
    entities['floors'] = [{'id': f.id(), 'name': f.Name, 'geometry': get_geometry(f)} for f in floors]
    
    # Extract spaces/rooms
    spaces = model.by_type("IfcSpace")
    entities['spaces'] = [{'id': s.id(), 'name': s.Name, 'geometry': get_geometry(s)} for s in spaces]
    
    # Extract furniture/objects (try different names)
    try:
        furniture = model.by_type("IfcFurniture")
    except:
        try:
            furniture = model.by_type("IfcFurnishingElement")
        except:
            furniture = []
    entities['furniture'] = [{'id': f.id(), 'name': f.Name, 'geometry': get_geometry(f)} for f in furniture]
    
    logger.info(f"Extracted entities: { {k: len(v) for k, v in entities.items()} }")
    return entities

def get_geometry(entity):
    """Extract geometry information from IFC entity (simplified)."""
    # In a full implementation, this would use ifcopenshell.geom for proper geometry extraction
    # For now, return placeholder
    return {'type': 'placeholder', 'bbox': None}

# Load IFC model
ifc_model = load_ifc_model(IFC_FILE)
ifc_entities = extract_building_entities(ifc_model)

INFO:__main__:Loaded IFC model
INFO:__main__:Extracted entities: {'walls': 13, 'floors': 3, 'spaces': 0, 'furniture': 71}


## 3. Extract Structural Priors from IFC

From the IFC entities, we derive topological priors such as room boundaries, spatial hierarchies, and connectivity constraints. This creates a prior knowledge graph that represents expected spatial relationships.

In [9]:
def build_ifc_prior_graph(entities):
    """Build a prior knowledge graph from IFC entities."""
    G = nx.Graph()
    
    # Add nodes for spaces/rooms
    for space in entities['spaces']:
        G.add_node(space['id'], type='space', name=space['name'], geometry=space['geometry'])
    
    # Add nodes for structural elements
    for wall in entities['walls']:
        G.add_node(wall['id'], type='wall', name=wall['name'], geometry=wall['geometry'])
    
    for floor in entities['floors']:
        G.add_node(floor['id'], type='floor', name=floor['name'], geometry=floor['geometry'])
    
    # Add containment relationships (spaces contain objects)
    # This is simplified; in practice, use IFC relationships
    for space in entities['spaces']:
        for furn in entities['furniture']:
            # Placeholder: assume furniture is in spaces based on proximity
            G.add_edge(space['id'], furn['id'], relation='contains')
    
    # Add adjacency relationships between spaces
    # Simplified: connect all spaces (in reality, check shared walls)
    space_ids = [s['id'] for s in entities['spaces']]
    for i in range(len(space_ids)):
        for j in range(i+1, len(space_ids)):
            G.add_edge(space_ids[i], space_ids[j], relation='adjacent')
    
    logger.info(f"Built IFC prior graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    return G

ifc_prior_graph = build_ifc_prior_graph(ifc_entities)

INFO:__main__:Built IFC prior graph with 16 nodes and 0 edges


## 4. Load and Process RGB-D Sequences

Load the egocentric RGB-D frame sequences, camera poses, and metadata. The data includes RGB images, depth maps, and camera trajectories.

In [10]:
def load_camera_poses(frames_path):
    """Load camera poses from frames.json."""
    with open(frames_path, 'r') as f:
        frames = json.load(f)
    poses = {}
    for frame in frames:
        pose = {
            'translation': np.array(frame['t_xyz_m']),
            'rotation': R.from_quat(frame['q_xyzw']).as_matrix(),
            'timestamp': frame['timestamp_s']
        }
        poses[frame['frame_id']] = pose
    logger.info(f"Loaded {len(poses)} camera poses")
    return poses

def load_camera_intrinsics(camera_info_path):
    """Load camera intrinsics."""
    with open(camera_info_path, 'r') as f:
        info = json.load(f)
    intrinsics = {
        'fx': info['intrinsics'][0][0],
        'fy': info['intrinsics'][1][1],
        'cx': info['intrinsics'][0][2],
        'cy': info['intrinsics'][1][2],
        'width': info['width'],
        'height': info['height']
    }
    return intrinsics

# Load data
frames_path = os.path.join(RGBD_DIR, 'frames.json')
camera_info_path = os.path.join(RGBD_DIR, 'camera_info.json')

camera_poses = load_camera_poses(frames_path)
camera_intrinsics = load_camera_intrinsics(camera_info_path)

# Update global intrinsics
CAMERA_INTRINSICS.update(camera_intrinsics)

INFO:__main__:Loaded 160 camera poses


## 5. 3D Point Cloud Generation and Preprocessing

Since point clouds are pre-computed, we load them directly. In a full implementation, this would involve converting RGB-D frames to 3D points using camera intrinsics and poses.

In [11]:
def load_point_clouds(pointcloud_dir, max_frames=10):
    """Load pre-computed point clouds."""
    pcd_files = sorted([f for f in os.listdir(pointcloud_dir) if f.endswith('.ply')])
    point_clouds = []
    
    for i, pcd_file in enumerate(pcd_files[:max_frames]):
        pcd_path = os.path.join(pointcloud_dir, pcd_file)
        pcd = o3d.io.read_point_cloud(pcd_path)
        point_clouds.append(pcd)
        logger.info(f"Loaded point cloud {i}: {len(pcd.points)} points")
    
    return point_clouds

pointcloud_dir = os.path.join(RGBD_DIR, 'pointcloud')
point_clouds = load_point_clouds(pointcloud_dir)

# Preprocessing: downsample and filter
def preprocess_point_cloud(pcd, voxel_size=0.05):
    """Downsample and remove outliers."""
    pcd_down = pcd.voxel_down_sample(voxel_size)
    pcd_down, ind = pcd_down.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    return pcd_down

processed_pcds = [preprocess_point_cloud(pcd) for pcd in point_clouds]

INFO:__main__:Loaded point cloud 0: 239890 points


## 6. Feature Extraction from RGB-D Frames

Extract visual and geometric features from the point clouds. This includes color, normals, and curvature for object detection and segmentation.

In [12]:
from pyexpat import features


# def extract_features(pcd):
#     """Extract geometric features from point cloud."""
#     # Estimate normals
#     pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    
#     # Compute FPFH features for geometric description
#     fpfh = o3d.pipelines.registration.compute_fpfh_feature(
#         pcd, o3d.geometry.KDTreeSearchParamHybrid(radius=0.25, max_nn=100))
    
#     features_ret = {
#         'points': np.asarray(pcd.points),
#         'colors': np.asarray(pcd.colors) if pcd.has_colors() else None,
#         'normals': np.asarray(pcd.normals),
#         'fpfh': np.asarray(fpfh.data)
#     }
#     # return features
#     for key, value in features.items():
#         if value is not None:
#             print(f"Feature: {key} | Shape: {value.shape}")
#         else:
#             print(f"Feature: {key} | Status: None")

# # Extract features for all point clouds
def extract_features(pcd):
    """Extract geometric features from point cloud."""
    # Estimate normals
    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    
    # Compute FPFH features for geometric description
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd, o3d.geometry.KDTreeSearchParamHybrid(radius=0.25, max_nn=100))
    
    features_ret = {
        'points': np.asarray(pcd.points),
        'colors': np.asarray(pcd.colors) if pcd.has_colors() else None,
        'normals': np.asarray(pcd.normals),
        'fpfh': np.asarray(fpfh.data)
    }
    return features_ret

# Extract features for all point clouds
point_cloud_features = [extract_features(pcd) for pcd in processed_pcds]

## 7. Visual Scene Graph Construction

Build an initial scene graph from visual observations. Nodes represent detected objects, edges represent spatial relationships.

In [13]:
def build_visual_scene_graph(features_list):
    """Build scene graph from visual features (simplified clustering-based)."""
    G = nx.Graph()
    
    for frame_idx, features in enumerate(features_list):
        # Simple clustering to detect "objects" (placeholder for actual object detection)
        from sklearn.cluster import DBSCAN
        points = features['points']
        clustering = DBSCAN(eps=0.5, min_samples=10).fit(points)
        labels = clustering.labels_
        unique_labels = set(labels)
        
        for label in unique_labels:
            if label == -1:  # Noise
                continue
            cluster_points = points[labels == label]
            centroid = np.mean(cluster_points, axis=0)
            
            node_id = f"obj_{frame_idx}_{label}"
            G.add_node(node_id, 
                      type='object', 
                      centroid=centroid, 
                      frame=frame_idx,
                      category='unknown')  # Placeholder category
    
    # Add spatial relationships
    nodes = list(G.nodes(data=True))
    for i, (n1, d1) in enumerate(nodes):
        for j, (n2, d2) in enumerate(nodes[i+1:], i+1):
            dist = np.linalg.norm(d1['centroid'] - d2['centroid'])
            if dist < 2.0:  # Proximity threshold
                G.add_edge(n1, n2, relation='near', distance=dist)
    
    logger.info(f"Built visual scene graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    return G

# # Extract features for all point clouds
# def build_visual_scene_graph(features_list):
#     """Build scene graph from visual features (simplified clustering-based)."""
#     G = nx.Graph()
    
#     for frame_idx, features in enumerate(features_list):
#         # Simple clustering to detect "objects" (placeholder for actual object detection)
#         points = features['points']
#         clustering = DBSCAN(eps=0.5, min_samples=10).fit(points)
#         labels = clustering.labels_
#         unique_labels = set(labels)
        
#         for label in unique_labels:
#             if label == -1:  # Noise
#                 continue
#             cluster_points = points[labels == label]
#             centroid = np.mean(cluster_points, axis=0)
            
#             node_id = f"obj_{frame_idx}_{label}"
#             G.add_node(node_id, 
#                       type='object', 
#                       centroid=centroid, 
#                       frame=frame_idx,
#                       category='unknown')  # Placeholder category
    
#     # Add spatial relationships
#     nodes = list(G.nodes(data=True))
#     for i, (n1, d1) in enumerate(nodes):
#         for j, (n2, d2) in enumerate(nodes[i+1:], i+1):
#             dist = np.linalg.norm(d1['centroid'] - d2['centroid'])
#             if dist < 2.0:  # Proximity threshold
#                 G.add_edge(n1, n2, relation='near', distance=dist)
    
#     logger.info(f"Built visual scene graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
#     return G

# Extract features for all point clouds
point_cloud_features = [extract_features(pcd) for pcd in processed_pcds]

# Build visual scene graph
visual_scene_graph = build_visual_scene_graph(point_cloud_features)

# # Build visual scene graph
# visual_scene_graph = build_visual_scene_graph(point_cloud_features)

INFO:__main__:Built visual scene graph with 1 nodes and 0 edges


## 10. IFC Prior Integration and Constraint Application

Fuse the visual scene graph with IFC priors. This involves aligning coordinate systems and applying structural constraints.

In [15]:
def fuse_scene_graphs(visual_graph, ifc_graph, transformation=None):
    """Fuse visual and IFC graphs (simplified alignment)."""
    fused_graph = nx.compose(visual_graph, ifc_graph)
    
    # Placeholder for coordinate alignment
    # In practice, estimate transformation between IFC and camera coordinate systems
    if transformation is None:
        transformation = np.eye(4)  # Identity for now
    
    # Apply constraints: e.g., objects should be inside spaces
    for node, data in visual_graph.nodes(data=True):
        if data['type'] == 'object':
            # Find nearest IFC space
            min_dist = float('inf')
            nearest_space = None
            for ifc_node, ifc_data in ifc_graph.nodes(data=True):
                if ifc_data['type'] == 'space':
                    # Placeholder distance calculation
                    dist = np.linalg.norm(data['centroid'])  # Simplified
                    if dist < min_dist:
                        min_dist = dist
                        nearest_space = ifc_node
            if nearest_space:
                fused_graph.add_edge(node, nearest_space, relation='inside', confidence=0.8)
    
    logger.info(f"Fused graph has {fused_graph.number_of_nodes()} nodes and {fused_graph.number_of_edges()} edges")
    return fused_graph

fused_scene_graph = fuse_scene_graphs(visual_scene_graph, ifc_prior_graph)

INFO:__main__:Fused graph has 17 nodes and 0 edges


## 13. Error Analysis and Failure Mode Discussion

### Potential Failure Modes

1. **Coordinate System Misalignment**: IFC models and RGB-D cameras may use different coordinate systems. Without proper alignment, spatial relationships will be incorrect.

2. **Dynamic Objects**: IFC models are static, but real environments contain moving objects (people, robots) that aren't in the BIM.

3. **Occlusion and Sensor Limitations**: RGB-D sensors have limited range and can be occluded, leading to incomplete point clouds.

4. **Semantic Ambiguity**: Visual features alone may not distinguish between similar objects (e.g., different types of furniture).

5. **Temporal Inconsistency**: Egocentric sequences may have motion blur or rapid movements affecting reconstruction quality.

### Mitigation Strategies

- Use ICP or other registration techniques for coordinate alignment.
- Implement temporal filtering and outlier rejection.
- Combine visual features with IFC priors for better semantic classification.
- Add uncertainty quantification to graph edges.

### Current Limitations of This Implementation

- Geometry extraction from IFC is placeholder; real implementation needs proper geometric processing.
- Object detection uses simple clustering instead of deep learning models.
- No actual fusion of coordinate systems.
- Limited to small number of frames for computational reasons.

This partial implementation demonstrates the architectural approach and highlights areas requiring further development.